# 02 - Análise Exploratória Completa

EDA dos 3.9M registros com gráficos interativos Plotly.

**Requisito:** Execute o notebook `01_data_loading.ipynb` primeiro.

In [ ]:
import sys
sys.path.insert(0, '.')
from config_analysis import *

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Tema escuro para todos os gráficos
import plotly.io as pio
pio.templates.default = 'plotly_dark'

# Cores neon
CYAN = '#00f0ff'
MAGENTA = '#ff00ff'
GREEN = '#00ff88'
ORANGE = '#ff8800'
RED = '#ff3366'
YELLOW = '#ffff00'

df = load_processed_data()
if df is None:
    raise FileNotFoundError("Execute o notebook 01 primeiro!")
print(f"Dataset: {len(df):,} registros, {len(df.columns)} colunas")

## 1. Visão Geral dos Multiplicadores

In [ ]:
# Estatísticas descritivas
stats = df['multiplicador'].describe(percentiles=[0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99])
print("=" * 40)
print("ESTATÍSTICAS DO MULTIPLICADOR")
print("=" * 40)
for name, val in stats.items():
    print(f"{name:>8s}: {val:>10.4f}")

print(f"\n% LOW (< {LOW_THRESHOLD}x): {(df['multiplicador'] < LOW_THRESHOLD).mean()*100:.2f}%")
print(f"% HIGH (>= {LOW_THRESHOLD}x): {(df['multiplicador'] >= LOW_THRESHOLD).mean()*100:.2f}%")

In [ ]:
# Histograma da distribuição dos multiplicadores (truncado em 20x para clareza)
df_hist = df[df['multiplicador'] <= 20]

fig = px.histogram(
    df_hist, x='multiplicador', nbins=200,
    title='Distribuição dos Multiplicadores (até 20x)',
    labels={'multiplicador': 'Multiplicador', 'count': 'Frequência'},
    color_discrete_sequence=[CYAN]
)
fig.add_vline(x=LOW_THRESHOLD, line_dash='dash', line_color=RED,
              annotation_text=f'LOW < {LOW_THRESHOLD}x')
fig.update_layout(height=500, bargap=0.02)
fig.show()

In [ ]:
# Boxplot comparativo LOW vs HIGH
fig = px.box(
    df[df['multiplicador'] <= 30], x='tipo', y='multiplicador',
    title='Boxplot: LOW vs HIGH',
    color='tipo',
    color_discrete_map={'LOW': RED, 'HIGH': GREEN}
)
fig.update_layout(height=500)
fig.show()

## 2. Análise Temporal

In [ ]:
# Média do multiplicador por hora do dia
hourly = df.groupby('hora').agg(
    media=('multiplicador', 'mean'),
    mediana=('multiplicador', 'median'),
    pct_low=('is_low', 'mean'),
    contagem=('multiplicador', 'count')
).reset_index()

fig = make_subplots(rows=2, cols=1, subplot_titles=(
    'Média do Multiplicador por Hora',
    '% de LOWs por Hora'
))

fig.add_trace(go.Bar(
    x=hourly['hora'], y=hourly['media'],
    name='Média', marker_color=CYAN
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=hourly['hora'], y=hourly['mediana'],
    name='Mediana', line=dict(color=MAGENTA, width=2)
), row=1, col=1)

fig.add_trace(go.Bar(
    x=hourly['hora'], y=hourly['pct_low'] * 100,
    name='% LOW', marker_color=RED
), row=2, col=1)
fig.add_hline(y=df['is_low'].mean()*100, line_dash='dash',
              line_color=YELLOW, row=2, col=1,
              annotation_text='Média global')

fig.update_layout(height=700, title_text='Padrões por Hora do Dia')
fig.show()

In [ ]:
# Média do multiplicador por dia da semana
dias = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
weekly = df.groupby('dia_semana').agg(
    media=('multiplicador', 'mean'),
    pct_low=('is_low', 'mean'),
    max_streak=('low_streak', 'max'),
    contagem=('multiplicador', 'count')
).reset_index()
weekly['dia_nome'] = [dias[i] for i in weekly['dia_semana']]

fig = make_subplots(rows=1, cols=2, subplot_titles=(
    'Média do Multiplicador', '% LOWs'
))

fig.add_trace(go.Bar(
    x=weekly['dia_nome'], y=weekly['media'],
    marker_color=CYAN, name='Média'
), row=1, col=1)

fig.add_trace(go.Bar(
    x=weekly['dia_nome'], y=weekly['pct_low'] * 100,
    marker_color=MAGENTA, name='% LOW'
), row=1, col=2)

fig.update_layout(height=450, title_text='Padrões por Dia da Semana')
fig.show()

In [ ]:
# Heatmap: Hora x Dia da Semana (% LOW)
heatmap_data = df.groupby(['dia_semana', 'hora'])['is_low'].mean().unstack(fill_value=0) * 100
heatmap_data.index = [dias[i] for i in heatmap_data.index]

fig = px.imshow(
    heatmap_data,
    title='% de LOWs por Hora x Dia da Semana',
    labels=dict(x='Hora', y='Dia', color='% LOW'),
    color_continuous_scale='RdYlGn_r',
    aspect='auto'
)
fig.update_layout(height=400)
fig.show()

## 3. Análise de Evolução Mensal

In [ ]:
# Evolução mensal
df['ano_mes'] = df['date'].dt.strftime('%Y-%m')
monthly = df.groupby('ano_mes').agg(
    media=('multiplicador', 'mean'),
    mediana=('multiplicador', 'median'),
    std=('multiplicador', 'std'),
    pct_low=('is_low', 'mean'),
    max_streak=('low_streak', 'max'),
    contagem=('multiplicador', 'count')
).reset_index()

fig = make_subplots(
    rows=3, cols=1,
    subplot_titles=('Média Mensal do Multiplicador', '% LOWs Mensal', 'Max Streak LOW Mensal'),
    shared_xaxes=True
)

fig.add_trace(go.Scatter(
    x=monthly['ano_mes'], y=monthly['media'],
    mode='lines+markers', name='Média',
    line=dict(color=CYAN, width=2)
), row=1, col=1)

fig.add_trace(go.Bar(
    x=monthly['ano_mes'], y=monthly['pct_low'] * 100,
    name='% LOW', marker_color=MAGENTA
), row=2, col=1)

fig.add_trace(go.Bar(
    x=monthly['ano_mes'], y=monthly['max_streak'],
    name='Max Streak', marker_color=RED
), row=3, col=1)
fig.add_hline(y=TRAGEDY_STREAK, line_dash='dash', line_color=YELLOW,
              row=3, col=1, annotation_text='Tragédia (12+)')

fig.update_layout(height=900, title_text='Evolução Mensal')
fig.show()

## 4. Distribuição de Streaks

In [ ]:
# Encontrar todas as sequências completas de LOWs
# Uma sequência termina quando is_low volta a 0
streak_ends = df[(df['low_streak'] > 0) & (df['is_low'].shift(-1, fill_value=0) == 0)]
streak_lengths = streak_ends['low_streak'].values

print(f"Total de sequências LOW encontradas: {len(streak_lengths):,}")
print(f"Comprimento médio: {streak_lengths.mean():.2f}")
print(f"Comprimento máximo: {streak_lengths.max()}")

# Distribuição observada vs teórica
p_low = df['is_low'].mean()  # Probabilidade empírica de LOW
max_len = int(streak_lengths.max())

observed = pd.Series(streak_lengths).value_counts().sort_index()
total_seqs = len(streak_lengths)

# Teórico: P(streak=k) = p^k * (1-p)
theoretical = {k: total_seqs * (p_low ** k) * (1 - p_low) for k in range(1, max_len + 1)}
theo_series = pd.Series(theoretical)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=observed.index[:20], y=observed.values[:20],
    name='Observado', marker_color=CYAN
))
fig.add_trace(go.Scatter(
    x=list(range(1, 21)), y=[theo_series.get(k, 0) for k in range(1, 21)],
    name='Teórico (iid)', mode='lines+markers',
    line=dict(color=RED, width=2, dash='dash')
))
fig.update_layout(
    title='Distribuição de Streaks LOW: Observado vs Teórico (IID)',
    xaxis_title='Comprimento da Sequência',
    yaxis_title='Frequência',
    height=500
)
fig.show()

# Tabela comparativa
print(f"\n{'Streak':>8} {'Observado':>12} {'Teórico':>12} {'Razão':>8}")
print("-" * 45)
for k in range(1, min(21, max_len + 1)):
    obs = observed.get(k, 0)
    teo = theo_series.get(k, 0)
    ratio = obs / teo if teo > 0 else 0
    marker = ' <<<' if abs(ratio - 1) > 0.1 else ''
    print(f"{k:>8d} {obs:>12,} {teo:>12,.0f} {ratio:>8.3f}{marker}")

## 5. Probabilidade Condicional por Posição na Streak

In [ ]:
# Para cada posição na streak, qual a probabilidade do PRÓXIMO ser LOW?
max_pos = 20
cond_probs = []

# Pré-calcular o is_low do PRÓXIMO round (no df original)
next_is_low = df['is_low'].shift(-1)

for pos in range(0, max_pos + 1):
    mask = df['low_streak'] == pos
    n = mask.sum()
    if n == 0:
        continue
    p = next_is_low[mask].mean()
    cond_probs.append({'posicao': pos, 'prob_next_low': p, 'amostra': n})

cp_df = pd.DataFrame(cond_probs)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=cp_df['posicao'], y=cp_df['prob_next_low'] * 100,
    marker_color=[GREEN if p < p_low else RED for p in cp_df['prob_next_low']],
    text=[f'{p*100:.1f}%' for p in cp_df['prob_next_low']],
    textposition='outside'
))
fig.add_hline(y=p_low * 100, line_dash='dash', line_color=YELLOW,
              annotation_text=f'Baseline: {p_low*100:.1f}%')
fig.update_layout(
    title='P(próximo=LOW) dado posição na streak LOW',
    xaxis_title='Posição na streak (0 = após HIGH)',
    yaxis_title='Probabilidade (%)',
    height=500
)
fig.show()

print("\nTabela de probabilidades condicionais:")
print(cp_df.to_string(index=False))

## 6. Correlação entre Features

In [ ]:
# Correlação entre features principais
feature_cols = [
    'multiplicador', 'is_low', 'low_streak', 'high_streak',
    'rolling_mean_5', 'rolling_mean_20', 'rolling_mean_100',
    'rolling_std_5', 'rolling_std_20', 'rolling_std_100',
    'pct_low_5', 'pct_low_20', 'pct_low_100',
    'lag_1', 'lag_5', 'log_return'
]

corr_matrix = df[feature_cols].corr()

fig = px.imshow(
    corr_matrix,
    title='Matriz de Correlação entre Features',
    color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1,
    text_auto='.2f',
    aspect='auto'
)
fig.update_layout(height=700, width=900)
fig.show()

## 7. Multiplicadores Extremos

In [ ]:
# Top 20 maiores multiplicadores
top20 = df.nlargest(20, 'multiplicador')[['date', 'multiplicador', 'hora', 'dia_semana_nome']]
print("TOP 20 - Maiores Multiplicadores:")
print(top20.to_string(index=False))

# Distribuição de multiplicadores extremos (>10x) por hora
extremos = df[df['multiplicador'] > 10]
fig = px.histogram(
    extremos, x='hora', nbins=24,
    title=f'Multiplicadores > 10x por Hora ({len(extremos):,} eventos)',
    color_discrete_sequence=[GREEN]
)
fig.update_layout(height=400)
fig.show()

## 8. Resumo dos Achados

In [ ]:
print("=" * 60)
print("RESUMO DA ANÁLISE EXPLORATÓRIA")
print("=" * 60)
print(f"""  
Dataset: {len(df):,} registros ({df['date'].min().strftime('%Y-%m')} a {df['date'].max().strftime('%Y-%m')})

Multiplicador:
  Média:   {df['multiplicador'].mean():.4f}x
  Mediana: {df['multiplicador'].median():.4f}x
  Std:     {df['multiplicador'].std():.4f}
  Max:     {df['multiplicador'].max():.1f}x

Classificação:
  LOW  (< {LOW_THRESHOLD}x): {df['is_low'].mean()*100:.2f}%
  HIGH (>= {LOW_THRESHOLD}x): {(1-df['is_low'].mean())*100:.2f}%

Streaks LOW:
  Total de sequências: {len(streak_lengths):,}
  Max streak: {streak_lengths.max()}
  Tragédias (12+): {df['is_tragedy'].sum()}

Probabilidade condicional (desvio do baseline {p_low*100:.1f}%):
  Após 5 LOWs: {cp_df[cp_df['posicao']==5]['prob_next_low'].values[0]*100:.1f}%
  Após 6 LOWs: {cp_df[cp_df['posicao']==6]['prob_next_low'].values[0]*100:.1f}%
  Após 8 LOWs: {cp_df[cp_df['posicao']==8]['prob_next_low'].values[0]*100:.1f}%
  Após 10 LOWs: {cp_df[cp_df['posicao']==10]['prob_next_low'].values[0]*100:.1f}%
""")

In [ ]:
import json
heatmap_data = df.groupby(['dia_semana', 'hora'])['is_low'].mean().unstack(fill_value=0) * 100
dias = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
heatmap_data.index = [dias[i] for i in heatmap_data.index]
print(heatmap_data.round(2).to_string())